# Video & Streaming

**TL;DR:** Expect "process this video": open it, read frames in a loop **with a
guard for failed reads**, do something per frame (motion, sample, annotate), and
**write an output video or frames**. The robust `cv2.VideoCapture` loop is the one
to have in muscle memory — it doubles as your fault-tolerance answer.

In [ ]:
%matplotlib inline
import cv2, numpy as np, matplotlib.pyplot as plt, os
from collections import deque

def show(img, title=""):
    """Display a BGR or grayscale image inline."""
    if img is None:
        print("image is None"); return
    plt.figure(figsize=(5,5))
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

def make_sample_assets():
    """Create synthetic inputs so every cell runs without your own files.
    Swap input.jpg for a real photo any time to experiment."""
    doc = np.full((700,500,3), 30, np.uint8)
    quad = np.array([[120,90],[420,140],[400,560],[90,520]], np.int32)
    cv2.fillConvexPoly(doc, quad, (235,235,235))
    cv2.putText(doc, "LABEL 12345", (150,300), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (20,20,20), 2)
    cv2.imwrite("input.jpg", doc)
    cv2.imwrite("noisy.jpg", doc)
    cv2.imwrite("low.jpg", (doc*0.3 + 90).astype(np.uint8))
    os.makedirs("ind", exist_ok=True)
    cv2.imwrite("ind/a.png", doc); cv2.imwrite("ind/b.png", doc)
    open("ind/c.png","wb").write(b"corrupt-not-an-image")
    vw = cv2.VideoWriter("test.avi", cv2.VideoWriter_fourcc(*"MJPG"), 10, (320,240))
    for i in range(12):
        fr = np.zeros((240,320,3), np.uint8)
        cv2.rectangle(fr, (10+i*15,80), (50+i*15,140), (255,255,255), -1)
        vw.write(fr)
    vw.release()
    # extra assets for the "More practice" problems (E1-E8)
    canvas = np.full((400,600,3), 60, np.uint8)
    cv2.rectangle(canvas, (80,100), (220,260), (60,180,60), -1)   # green box (BGR)
    cv2.circle(canvas, (430,180), 70, (60,60,200), -1)            # red circle
    cv2.imwrite("color.jpg", canvas)
    bc = np.full((400,600,3), 200, np.uint8)
    x, rng = 150, np.random.default_rng(0)
    while x < 450:                        # random-width vertical bars = fake barcode
        bw = int(rng.integers(3, 10))
        cv2.rectangle(bc, (x,140), (x+bw,260), (0,0,0), -1)
        x += bw + int(rng.integers(3, 10))
    cv2.imwrite("barcode.jpg", bc)
    tbl = np.full((400,600,3), 255, np.uint8)
    for gx in range(50, 601, 110):        # vertical table lines
        cv2.line(tbl, (gx,40), (gx,360), (0,0,0), 2)
    for gy in range(40, 401, 80):         # horizontal table lines
        cv2.line(tbl, (50,gy), (490,gy), (0,0,0), 2)
    cv2.imwrite("lines.jpg", tbl)
    return doc

doc = make_sample_assets()
print("Sample assets ready: input.jpg, noisy.jpg, low.jpg, color.jpg, barcode.jpg, lines.jpg, ind/, test.avi")
show(doc, "synthetic input.jpg  (replace with your own image!)")


> **Tip:** run the setup cell above first. Each problem below defines its solution and then runs a quick demo. Edit and re-run any cell to experiment.

## Problem 13 — Robust capture: read, sample every Nth frame, save

**Tests:** the capture loop, failure handling, frame sampling, releasing resources.

**The problem:** open a video (file or RTSP stream), keep every Nth frame as an
image, and don't crash or leak when the source misbehaves.

**The plan:**

```text
 frames:   0   1  ...  14  [15]  16 ...  29  [30]     every=15
          save                save            save

 the loop that matters:
    while True:  ok, frame = cap.read()
                 if not ok: break         <- EOF *or* dropped stream
    finally:     cap.release()            <- runs no matter what
```

**Why this way:** count-and-modulo beats seeking — jumping around with
`CAP_PROP_POS_FRAMES` is unreliable on many codecs (seeks snap to keyframes),
while sequential reading is always exact. The `ok` guard and `finally: release`
ARE the fault-tolerance answer: streams end without warning, and leaked capture
handles are how long-running services die slowly. For live RTSP, wrap the whole
thing in reconnect-with-backoff.

In [ ]:
import cv2
import os

def sample_frames(source, out_dir="frames", every=15):
    """source: a video path or RTSP url. Saves every Nth frame as an image."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {source}")
    frame_index = saved_count = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:                       # end of file OR dropped stream
                break
            if frame_index % every == 0:
                filename = os.path.join(out_dir, f"frame_{frame_index:05d}.jpg")
                cv2.imwrite(filename, frame)
                saved_count += 1
            frame_index += 1
    finally:
        cap.release()                        # ALWAYS release (no leaks)
    print(f"read {frame_index} frames, saved {saved_count}")
    return saved_count

In [ ]:
print("saved", sample_frames("test.avi","frames",3), "frames"); show(cv2.imread("frames/frame_00000.jpg"), "a sampled frame")

**Watch out:** check `isOpened()`, handle `ok == False` (don't assume frames keep
coming), and `release()` in a `finally`. For live RTSP you'd wrap this in a
reconnect-with-backoff loop (see the fault-tolerance chapter).

---

## Problem 14 — Motion detection via frame differencing, write annotated video

**Tests:** per-frame processing, background diff, contours on a mask, `VideoWriter`.

**The problem:** find and box the *moving* things in a video, writing an
annotated copy.

**The plan:**

```text
 prev frame    cur frame     absdiff      threshold     dilate+contours
 [.......]  vs [...o...] ==> [...#...] ==> [...#...] ==> [..[box]..]
 (blur both first - otherwise sensor noise "moves" in every pixel)
```

**Why this way:** frame differencing needs zero training and one frame of
state — the right first answer. Know its weakness out loud: it detects *change*,
so an object that stops moving disappears. The production upgrade is
`cv2.createBackgroundSubtractorMOG2()`, which learns the background over time
and tolerates gradual lighting change. Practical trap: `VideoWriter` silently
produces an empty file if the frame size doesn't match — always pass the real
width/height.

In [ ]:
import cv2

def detect_motion(source, out="p14_out.mp4", min_area=800):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {source}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25            # some sources report 0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")         # codec for .mp4
    writer = cv2.VideoWriter(out, fourcc, fps, (width, height))
    prev_gray = None                                 # nothing to diff on frame 1
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY),
                                    (21, 21), 0)
            if prev_gray is not None:
                frame_diff = cv2.absdiff(prev_gray, gray)   # |previous - current|
                motion_mask = cv2.threshold(frame_diff, 25, 255,
                                            cv2.THRESH_BINARY)[1]  # keep big changes
                motion_mask = cv2.dilate(motion_mask, None, iterations=2)  # merge blobs
                contours, _ = cv2.findContours(motion_mask, cv2.RETR_EXTERNAL,
                                               cv2.CHAIN_APPROX_SIMPLE)
                for contour in contours:
                    if cv2.contourArea(contour) < min_area:  # ignore tiny flicker
                        continue
                    x, y, box_w, box_h = cv2.boundingRect(contour)
                    cv2.rectangle(frame, (x, y), (x + box_w, y + box_h),
                                  (0, 255, 0), 2)
            writer.write(frame)
            prev_gray = gray                 # current frame becomes the baseline
    finally:
        cap.release(); writer.release()      # ALWAYS release both (no leaks)
    return out

In [ ]:
detect_motion("test.avi","motion.avi"); print("wrote motion.avi")

**Watch out:** `VideoWriter` size must match the frames you write, or you get an
empty file. Blur before diff to kill sensor noise. `fourcc` "mp4v" for `.mp4`.

---

## Problem 15 — Overlay FPS / text and count frames

**Tests:** drawing text, reading video properties, simple per-frame overlay.

**The problem:** overlay live diagnostics (frame number, throughput) on each
frame — the "prove your pipeline keeps up" task.

**The plan:**

```text
 read frame -> putText("frame N   X fps") -> writer.write(frame)

 measured fps = frames processed / wall-clock time
 (NOT the file's metadata fps - that's just its playback rate)
```

**Why this way:** measured fps is the number that answers "is my processing
real-time?" — comparing it against the source fps tells you if you're falling
behind. The drawing gotchas to memorize: `putText` coordinates are the
**bottom-left** of the text (not top-left), coordinates are (x, y), and colors
are BGR — yellow is `(0, 255, 255)`.

In [ ]:
import cv2, time

def annotate(source, out="p15_out.mp4"):
    cap = cv2.VideoCapture(source)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    width, height = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                     int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), fps,
                             (width, height))
    frame_count, start_time = 0, time.time()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_count += 1
        measured_fps = frame_count / (time.time() - start_time + 1e-9)  # avoid /0
        cv2.putText(frame, f"frame {frame_count}  {measured_fps:4.1f} fps",
                    (10, 30),                        # (x, y) of text bottom-left
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)  # BGR yellow
        writer.write(frame)
    cap.release(); writer.release()
    return frame_count

In [ ]:
print("frames annotated:", annotate("test.avi","annot.avi"))

**Watch out:** `putText` origin is the **bottom-left** of the text and coords are
(x, y). Colors are **BGR**.